<h1><img src="../../../icons/tk_full_logo.svg" width="80" /> Fine-Tune GPT-OSS 20B for Tool Use</h1>

Fine-tune GPT-OSS 20B with **Unsloth QLoRA** (~14GB VRAM).

**Prerequisites:**
- Run `04a-generate-training-data.ipynb` first
- Model `unsloth/gpt-oss-20b-bnb-4bit` mirrored to MLflow
- GPU with 16GB+ VRAM

---
## 1. Setup

In [1]:
import os
import json
import torch
from pathlib import Path

from check_jupyter_flavor import check_flavor
check_flavor('fine-tuning')

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

max_seq_length = 2048  # Increased for harmony format overhead

✓ Running in correct environment: fine-tuning
  Fine-Tuning Lab (Unsloth, QLoRA, PEFT, TRL + ml-gpu)
CUDA: True
GPU: NVIDIA GB10
Memory: 128.5 GB


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(


---
## 2. Load Training Data

In [2]:
from datasets import Dataset

# Try checkpoint first, then output file
DATA_PATH = Path("tool_use_checkpoint.json")
if not DATA_PATH.exists():
    DATA_PATH = Path("tool_use_training_data.json")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Run 04a-generate-training-data.ipynb first!")

with open(DATA_PATH) as f:
    training_data = json.load(f)

print(f"Loaded: {len(training_data)} examples from {DATA_PATH}")

# Show distribution - extract function name from 'to' field
def get_function_name(example):
    """Extract function name from assistant message."""
    for msg in example["messages"]:
        if msg.get("role") == "assistant" and "to" in msg:
            # Extract function name from "functions.search_papers" format
            return msg["to"].split(".")[-1]
    return None

search_count = sum(1 for e in training_data if get_function_name(e) == "search_papers")
details_count = sum(1 for e in training_data if get_function_name(e) == "get_paper_details")
mlflow_count = sum(1 for e in training_data if get_function_name(e) == "log_to_mlflow")

print(f"\nDistribution:")
print(f"  - search_papers: {search_count}")
print(f"  - get_paper_details: {details_count}")
print(f"  - log_to_mlflow: {mlflow_count}")

# Preview
print(f"\nSample:")
user_msg = next(m for m in training_data[0]["messages"] if m["role"] == "user")
asst_msg = next(m for m in training_data[0]["messages"] if m["role"] == "assistant")
print(f"User: {user_msg['content']}")
print(f"Tool: {asst_msg['to']} -> {asst_msg['content'][:80]}...")

Loaded: 2000 examples from tool_use_checkpoint.json

Distribution:
  - search_papers: 993
  - get_paper_details: 572
  - log_to_mlflow: 435

Sample:
User: Can you find research about nanomaterial properties in materials science?
Tool: functions.search_papers -> {"query": "text='nanomaterial mechanical and electronic property characterizatio...


---
## 3. Load Model

In [3]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"


def get_model_path(model_id: str) -> Path:
    """Get model path from MLflow."""
    model_name = model_id.replace('/', '-')
    mlflow_url = os.environ.get('MLFLOW_TRACKING_URI')
    token_url = os.environ.get('MLFLOW_KEYCLOAK_TOKEN_URL')
    
    token = None
    if token_url:
        resp = requests.post(token_url, data={
            'grant_type': 'password',
            'client_id': os.environ.get('MLFLOW_KEYCLOAK_CLIENT_ID', 'mlflow'),
            'client_secret': os.environ.get('MLFLOW_CLIENT_SECRET'),
            'username': os.environ.get('MLFLOW_AUTH_USERNAME'),
            'password': os.environ.get('MLFLOW_AUTH_PASSWORD'),
            'scope': 'openid'
        }, verify=False, timeout=30)
        resp.raise_for_status()
        token = resp.json()['access_token']
    
    headers = {'Authorization': f'Bearer {token}'} if token else {}
    
    resp = requests.get(
        f"{mlflow_url}/api/2.0/mlflow/model-versions/search",
        params={'filter': f"name='{model_name}'"},
        headers=headers, verify=False, timeout=30
    )
    resp.raise_for_status()
    
    versions = resp.json().get('model_versions', [])
    if not versions:
        raise ValueError(f"Model '{model_name}' not found in MLflow")
    
    latest = max(versions, key=lambda v: int(v['version']))
    run_id = latest['run_id']
    
    resp = requests.get(
        f"{mlflow_url}/api/2.0/mlflow/runs/get",
        params={'run_id': run_id},
        headers=headers, verify=False, timeout=30
    )
    resp.raise_for_status()
    experiment_id = resp.json()['run']['info']['experiment_id']
    
    for base in [Path('/home/jovyan/thinkube/mlflow'), Path.home() / 'thinkube' / 'mlflow']:
        path = base / 'artifacts' / experiment_id / run_id / 'artifacts' / 'model'
        if path.exists():
            return path
    
    raise FileNotFoundError("Model not found")


MODEL_ID = "unsloth/gpt-oss-20b-bnb-4bit"
model_path = get_model_path(MODEL_ID)
print(f"Model path: {model_path}")

Model path: /home/jovyan/thinkube/mlflow/artifacts/1/e0d35d5fb9eb4f5a8d4a26119a0dc6d4/artifacts/model


In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(model_path),
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
    trust_remote_code=True,
    device_map={"": 0},
)

print(f"Loaded: {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"GPU Memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not import trl.trainer.alignprop_trainer: Failed to import trl.trainer.alignprop_trainer because of the following error (look up to see its traceback):
cannot import name 'DDPOStableDiffusionPipeline' from 'trl.models' (/usr/local/lib/python3.12/dist-packages/trl/models/__init__.py)


[unsloth_zoo.log|WARNING]Unsloth: Failed to import trl openenv: No module named 'trl.experimental'


Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.12.1: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 119.697 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gpt_Oss does not support SDPA - switching to fast eager.


Loading checkpoint shards:   0%|          | 0/9 [00:00<?, ?it/s]

Loaded: 20,596,252,224 parameters
GPU Memory: 40.9 GB


---
## 4. Format Data for Training

Convert OpenAI tool-use format to GPT-OSS Harmony chat format.

In [ ]:
!pip install openai-harmony

In [5]:
from openai_harmony import load_harmony_encoding, HarmonyEncodingName

# Load harmony encoding for GPT-OSS
harmony_enc = load_harmony_encoding(HarmonyEncodingName.HARMONY_GPT_OSS)

def format_for_harmony(example):
    """Convert training data to proper GPT-OSS Harmony format.
    
    The training data from 04a has:
    - developer message with tool declarations
    - user message with query
    - assistant message with channel='commentary', to='functions.xxx', content=JSON args
    
    We need to format this as proper harmony tokens for training.
    """
    messages = example["messages"]
    
    # Find messages by role
    developer_msg = next((m for m in messages if m["role"] == "developer"), None)
    user_msg = next(m for m in messages if m["role"] == "user")
    asst_msg = next(m for m in messages if m["role"] == "assistant")
    
    # Build the prompt part (system + user)
    # Use tokenizer's chat template for the prompt portion
    prompt_messages = []
    if developer_msg:
        prompt_messages.append({"role": "system", "content": developer_msg["content"]})
    prompt_messages.append({"role": "user", "content": user_msg["content"]})
    
    prompt = tokenizer.apply_chat_template(
        prompt_messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # Build the assistant response in proper harmony format
    # Tool calls use 'commentary' channel with 'to' field
    channel = asst_msg.get("channel", "commentary")
    to_field = asst_msg.get("to", "")
    content = asst_msg.get("content", "")
    
    # Construct harmony-formatted assistant response
    # Format: <|start|>assistant<|channel|>commentary<|to|>functions.search_papers<|message|>{"args"}<|end|>
    if to_field:
        assistant_response = f"<|start|>assistant<|channel|>{channel}<|to|>{to_field}<|message|>{content}<|end|>"
    else:
        assistant_response = f"<|start|>assistant<|channel|>{channel}<|message|>{content}<|end|>"
    
    # Combine prompt and response
    # Remove the trailing generation prompt marker from tokenizer output
    if prompt.endswith("<|start|>assistant<|message|>"):
        prompt = prompt[:-len("<|start|>assistant<|message|>")]
    elif prompt.endswith("<|start|>assistant"):
        prompt = prompt[:-len("<|start|>assistant")]
    
    full_text = prompt + assistant_response
    
    return {"text": full_text}


# Convert all examples
formatted_data = [format_for_harmony(ex) for ex in training_data]
train_dataset = Dataset.from_list(formatted_data)
train_dataset = train_dataset.filter(lambda x: len(x["text"]) > 0)

print(f"Formatted: {len(train_dataset)} examples")
print(f"\nSample (showing harmony structure):")
sample = train_dataset[0]["text"]
# Show the assistant part clearly
if "<|start|>assistant" in sample:
    asst_start = sample.index("<|start|>assistant")
    print(f"...{sample[asst_start:]}")

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Formatted: 2000 examples

Sample (showing harmony structure):
...<|start|>assistant<|channel|>commentary<|to|>functions.search_papers<|message|>{"query": "text='nanomaterial mechanical and electronic property characterization in materials science"}<|end|>


---
## 5. Add LoRA

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} ({100*trainable/total:.3f}%)")

Unsloth: Making `model.base_model.model.model` require gradients
Trainable: 3,981,312 (0.019%)


---
## 6. Train

In [7]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

sft_config = SFTConfig(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=180,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir="outputs",
    report_to="none",
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=sft_config,
)

# Train on responses only - match the harmony format we're using
# The response starts at <|start|>assistant<|channel|>commentary
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start|>user<|message|>",
    response_part="<|start|>assistant<|channel|>commentary"
)

print("Training...")
stats = trainer.train()
print(f"\nDone! Loss: {stats.training_loss:.4f}")

Unsloth: Tokenizing ["text"] (num_proc=24):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=24):   0%|          | 0/2000 [00:00<?, ? examples/s]

Training...


Step,Training Loss
1,4.925400
2,4.683700
3,3.639800
4,3.135700
5,3.350000
6,3.159700
7,2.345200
8,1.642500
9,1.488700
10,1.169200



Done! Loss: 0.3639


---
## 7. Test

In [9]:
from openai_harmony import Role

FastLanguageModel.for_inference(model)

def parse_harmony_response(text: str) -> dict:
    """Parse harmony format response with error handling."""
    result = {"tool_calls": [], "final": None, "raw": text}
    
    try:
        tokens = harmony_enc.encode(text, allowed_special="all")
        messages = harmony_enc.parse_messages_from_completion_tokens(
            tokens, role=Role.ASSISTANT, strict=False
        )
        
        for msg in messages:
            channel = getattr(msg, 'channel', None)
            to_field = getattr(msg, 'to', None)
            content = msg.content
            if isinstance(content, list):
                content = ''.join(str(c) for c in content)
            
            if channel == 'commentary' and to_field:
                result["tool_calls"].append({
                    "function": to_field,
                    "arguments": content
                })
            elif channel == 'final':
                result["final"] = content
    except Exception as e:
        import re
        
        to_match = re.search(r'<\|to\|>([^<]+)', text)
        msg_match = re.search(r'<\|message\|>([^<]+)', text)
        
        if to_match and msg_match:
            result["tool_calls"].append({
                "function": to_match.group(1),
                "arguments": msg_match.group(1)
            })
        else:
            asst_match = re.search(r'<\|start\|>assistant.*?<\|message\|>([^<]+)', text, re.DOTALL)
            if asst_match:
                result["final"] = asst_match.group(1)
    
    return result


test_prompts = [
    "Find papers about transformer architectures",
    "Search for research on protein folding",
    "Log my analysis to MLflow",
    "Get details on the BERT paper",
    "What papers exist about reinforcement learning?",
]

print("Testing:\n")

for prompt in test_prompts:
    system_msg = """You are a research assistant that helps users find and analyze papers.

namespace functions {
  type search_papers = (_: { query: string }) => any;
  type get_paper_details = (_: { paper_title_fragment: string }) => any;
  type log_to_mlflow = (_: { experiment_name: string; run_name: string; findings_summary: string }) => any;
}"""
    
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt}
    ]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(response_tokens, skip_special_tokens=False)
    
    parsed = parse_harmony_response(response)
    
    print(f"[USER] {prompt}")
    if parsed["tool_calls"]:
        for tc in parsed["tool_calls"]:
            print(f"[TOOL] {tc['function']}({tc['arguments']})")
    elif parsed["final"]:
        print(f"[FINAL] {parsed['final'][:100]}...")
    else:
        print(f"[RAW] {response[:100]}...")
    print("-" * 50)

Testing:

[USER] Find papers about transformer architectures
[TOOL] functions.search_papers({"query": "text='transformer architectures research papers"})
--------------------------------------------------
[USER] Search for research on protein folding
[FINAL] text='Below is a curated list of recent research papers on protein folding, along with'...
--------------------------------------------------
[USER] Log my analysis to MLflow
[TOOL] functions.search_papers({"experiment_name": "machine-learning-research", "run_name": "analysis-run", "findings_summary": "Analysis of machine learning research papers"})
--------------------------------------------------
[USER] Get details on the BERT paper
[FINAL] text='Here are the details of the BERT paper:\n\n**Title:**  \n*BERT: Pre-training of Deep Bidirecti...
--------------------------------------------------
[USER] What papers exist about reinforcement learning?
[TOOL] functions.get_paper_details({"query": "text='reinforcement learning research

---
## 8. Save

In [ ]:
OUTPUT_DIR = Path("/home/jovyan/thinkube/models/gpt-oss-tool-use")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# LoRA adapters
LORA_DIR = OUTPUT_DIR / "lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA: {LORA_DIR}")

# Merged model
MERGED_DIR = OUTPUT_DIR / "merged"
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print(f"Merged: {MERGED_DIR}")

---
## Done!

The fine-tuned model is saved and ready for deployment with CrewAI, LangChain, or other orchestration systems.